# PaddleOCR-VL 0.9B — Test propre sur Domino

Objectif : valider **uniquement le modèle PaddleOCR-VL 0.9B en local** avant de l'intégrer au pipeline OCR bancaire.

Ordre des tests :
1. environnement ;
2. recherche automatique du modèle dans ModelHub ;
3. contrôle des fichiers ;
4. chargement offline ;
5. conversion PDF → image ;
6. OCR d'une seule page ;
7. tests Table / Formula / Chart.

**Ne pas lancer le pipeline V12 dans ce notebook.**


## 1. Environnement


In [ ]:
import os
import sys
import time
from pathlib import Path

import torch
import fitz
from PIL import Image
import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText

print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA         :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version :", torch.version.cuda)
    print("GPU          :", torch.cuda.get_device_name(0))
    print("BF16 support :", torch.cuda.is_bf16_supported())

assert torch.cuda.is_available(), "GPU CUDA requis pour ce test."


## 2. Configuration fixe du modèle


In [ ]:
# Configuration fixe Domino
MODEL_PATH = Path("/domino/edv/modelhub/ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL/main")

DEVICE = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

PDF_ZOOM = 2.5
IMAGE_MAX_SIZE = 1800
MAX_NEW_TOKENS = 1200

print("MODEL_PATH :", MODEL_PATH)
print("Existe     :", MODEL_PATH.exists())
print("Device     :", DEVICE)
print("Dtype      :", DTYPE)

assert MODEL_PATH.is_dir(), f"Modèle introuvable : {MODEL_PATH}"

print("\nFichiers du modèle :")
for name in sorted(p.name for p in MODEL_PATH.iterdir())[:50]:
    print(" -", name)


## 3. Vérification minimale du dépôt


In [ ]:
assert MODEL_PATH is not None, "Définir MODEL_PATH."
assert MODEL_PATH.is_dir(), f"Répertoire introuvable : {MODEL_PATH}"

required_or_expected = [
    "config.json",
    "preprocessor_config.json",
]

print("Contrôle :")
for name in required_or_expected:
    path = MODEL_PATH / name
    print(f"{name:30s} :", "OK" if path.exists() else "ABSENT")

weight_files = (
    list(MODEL_PATH.glob("*.safetensors"))
    + list(MODEL_PATH.glob("*.bin"))
)

print("Fichiers de poids             :", len(weight_files))
assert (MODEL_PATH / "config.json").exists(), "config.json absent."
assert weight_files, "Aucun fichier de poids trouvé."

print("\nDépôt local exploitable.")


## 4. Configuration d'inférence


## 4. Chargement PaddleOCR-VL — offline

`local_files_only=True` empêche Transformers d'essayer de télécharger le modèle depuis Hugging Face.


In [ ]:
print("Chargement du processor...")
t0 = time.time()

processor = AutoProcessor.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
)

print("Processor chargé :", type(processor))

print("\nChargement du modèle...")

model = AutoModelForImageTextToText.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
    dtype=DTYPE,
    device_map="auto",
)

model.eval()

print(f"\nModèle chargé en {time.time() - t0:.1f} s")
print("Type modèle :", type(model))
print("Device      :", next(model.parameters()).device)
print("Dtype       :", next(model.parameters()).dtype)
print("VRAM        :", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


## 5. Fonctions PDF → image


In [ ]:
def resize_image(image, max_side=IMAGE_MAX_SIZE):
    w, h = image.size

    if max(w, h) <= max_side:
        return image

    ratio = max_side / max(w, h)

    return image.resize(
        (int(w * ratio), int(h * ratio)),
        Image.LANCZOS
    )


def pdf_page_to_image(pdf_path, page_index=0, zoom=PDF_ZOOM):
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(pdf_path)

    with fitz.open(str(pdf_path)) as doc:
        if page_index < 0 or page_index >= len(doc):
            raise IndexError(
                f"Page {page_index} invalide. PDF = {len(doc)} page(s)."
            )

        page = doc.load_page(page_index)

        pix = page.get_pixmap(
            matrix=fitz.Matrix(zoom, zoom),
            alpha=False
        )

        image = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

    return resize_image(image)


print("Fonctions PDF prêtes.")


## 6. Fonction d'inférence PaddleOCR-VL

Pour le premier diagnostic, on utilise les commandes natives du modèle plutôt qu'un prompt métier complexe.


In [ ]:
TASK_PROMPTS = {
    "ocr": "OCR:",
    "table": "Table Recognition:",
    "formula": "Formula Recognition:",
    "chart": "Chart Recognition:",
}


def paddle_ocr(image, task="ocr", max_new_tokens=MAX_NEW_TOKENS):
    if task not in TASK_PROMPTS:
        raise ValueError(
            f"Tâche inconnue : {task}. "
            f"Valeurs possibles : {list(TASK_PROMPTS)}"
        )

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": TASK_PROMPTS[task]},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    model_device = next(model.parameters()).device
    inputs = {
        key: value.to(model_device) if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    t0 = time.time()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    input_length = inputs["input_ids"].shape[-1]
    generated = outputs[0][input_length:]

    text = processor.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return {
        "text": text,
        "tokens_out": int(generated.numel()),
        "elapsed_s": round(time.time() - t0, 3),
    }


print("Fonction d'inférence prête.")


## 7. Choisir un PDF de test

Modifier uniquement `TEST_PDF`.

Commencer par un document court et lisible afin de valider le modèle avant les scans difficiles.


In [ ]:
TEST_PDF = Path("/mnt/data/transferts_in/document_test.pdf")
TEST_PAGE = 0

print("PDF       :", TEST_PDF)
print("Existe    :", TEST_PDF.exists())
print("Page test :", TEST_PAGE)

if TEST_PDF.exists():
    with fitz.open(str(TEST_PDF)) as doc:
        print("Nb pages  :", len(doc))


## 8. Afficher la page test


In [ ]:
assert TEST_PDF.exists(), f"PDF introuvable : {TEST_PDF}"

image = pdf_page_to_image(
    TEST_PDF,
    page_index=TEST_PAGE
)

print("Dimensions :", image.size)
display(image)


## 9. Test OCR brut


In [ ]:
result = paddle_ocr(
    image,
    task="ocr"
)

print("=" * 80)
print("OCR BRUT PADDLEOCR-VL")
print("=" * 80)
print(result["text"])
print("=" * 80)

print("Temps      :", result["elapsed_s"], "s")
print("Tokens out :", result["tokens_out"])


## 10. Tests spécialisés optionnels

Ne lancer que la cellule correspondant au contenu de la page.


In [ ]:
# TABLE
# result_table = paddle_ocr(image, task="table")
# print(result_table["text"])

# FORMULE
# result_formula = paddle_ocr(image, task="formula")
# print(result_formula["text"])

# GRAPHIQUE
# result_chart = paddle_ocr(image, task="chart")
# print(result_chart["text"])


## 11. Contrôle final

À ce stade, le test doit répondre à trois questions :

- le modèle est-il correctement trouvé dans ModelHub ?
- le modèle se charge-t-il entièrement sans accès Internet ?
- l'OCR brut reproduit-il correctement le contenu réel du document ?

On ne réintègre la classification, les prompts métier, JSON, batch GPU et Excel du pipeline V12 **qu'après validation de ce test minimal**.

Cette séparation évite de confondre une erreur de modèle/processor avec une erreur du pipeline métier.
